# 🏦 FairTransCP: Fair Conformal Classification for Financial Transactions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarushdubey/fair-conformal-finance/blob/main/notebooks/demo.ipynb)
[![Paper Under Review](https://img.shields.io/badge/AISTATS-2027-blue.svg)](https://github.com/aarushdubey/fair-conformal-finance)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

**Author:** Aarush Dubey  
**Conference Submission:** AISTATS 2027 (Artificial Intelligence and Statistics)  
**Paradigm:** Purely Classical Machine Learning (Random Forests, XGBoost, LightGBM) — Zero Deep Learning  

---
### 🎯 The Problem We Solve: The Coverage-Equity Paradox
In consumer credit scoring, automated systems must provide honest uncertainty quantification. Conformal Prediction guarantees that true labels are covered at a pre-specified safety level (e.g. 90%).

However, previous fairness methods naively force equal coverage across groups. On protected groups with fewer data points or higher noise, the model takes the easy way out: **it outputs massive, ambiguous prediction sets `{Approve, Reject}`**. In real banking, ambiguous applicants face endless paperwork, audit delays, or informal rejection.

**FairTransCP** jointly optimizes **coverage equity** AND **set-size equity**, preventing this discrimination.

---
### 🚀 1-Click Run
Press **Shift + Enter** through each cell below or click **Runtime → Run all** to test the entire framework in the cloud with zero local setup!

In [ ]:
# Cell 1: Cloud Environment Detection and Setup
import sys
import os

if 'google.colab' in sys.modules:
    print('⚡ Running in Google Colab! Cloning repository and installing dependencies...')
    !git clone -q https://github.com/aarushdubey/fair-conformal-finance.git
    %cd fair-conformal-finance
    !pip install -q ucimlrepo xgboost lightgbm
    sys.path.insert(0, '.')
else:
    print('💻 Running locally.')
    sys.path.insert(0, os.path.abspath('..'))

print('✓ Environment configured successfully!')

### 📊 Step 1: Load Benchmark Financial Data
We load the UCI German Credit benchmark (1,000 applicants with demographic attributes and loan repayment labels).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data.loaders import load_dataset

data = load_dataset('german_credit', sensitive='gender')
X, y, sens = data['X'], data['y'], data['sensitive']

groups, counts = np.unique(sens, return_counts=True)
print(f"Dataset: {data['description']}")
print(f"Feature matrix shape: {X.shape}")
print(f"Total applicants: {len(y)}")
print(f"Demographic representation: {dict(zip(groups, counts))}")

### 🌲 Step 2: Fit Classical ML Model
We partition data into **Training (50%)**, **Calibration (25%)**, and **Testing (25%)** splits.
We train an auditable Random Forest classifier (fully compliant with financial regulations like FCRA and ECOA).

In [ ]:
from sklearn.model_selection import train_test_split
from src.models.classifiers import get_classifier

# 3-way split: Train, Calibration, Test
X_trainval, X_test, y_trainval, y_test, s_trainval, s_test = train_test_split(
    X, y, sens, test_size=0.25, random_state=42, stratify=y
)
X_train, X_cal, y_train, y_cal, s_train, s_cal = train_test_split(
    X_trainval, y_trainval, s_trainval, test_size=0.33, random_state=42, stratify=y_trainval
)

clf = get_classifier('rf', random_state=42)
clf.fit(X_train, y_train)

prob_cal = clf.predict_proba(X_cal)
prob_test = clf.predict_proba(X_test)
print(f"✓ Trained Random Forest on {len(X_train)} samples.")
print(f"✓ Reserved {len(X_cal)} calibration samples and {len(X_test)} test evaluation samples.")

### ⚖️ Step 3: Run Conformal Calibration Comparison
We calibrate all three methods targeting a **90% guaranteed coverage level** ($\alpha = 0.10$):
1. **Standard Conformal Prediction:** Marginal calibration (ignores group membership).
2. **Naive Group-Conditional CP:** Calibrates per group separately (inflates set sizes on protected groups).
3. **FairTransCP (Our Method):** Jointly balances coverage equity and set-size parity.

In [ ]:
from src.conformal.base import SplitConformalClassifier
from src.conformal.fair_conformal import FairTransCP
from src.fairness.metrics import compute_all_metrics

alpha = 0.10  # 90% target coverage

# 1. Standard Conformal Prediction
std_cp = SplitConformalClassifier(alpha=alpha, score_fn='softmax')
std_cp.calibrate(prob_cal, y_cal)
sets_std = std_cp.predict_sets(prob_test)
m_std = compute_all_metrics(sets_std, y_test, s_test)

# 2. Naive Group-Conditional CP
grp_cp = FairTransCP(alpha=alpha, score_fn='softmax', fairness_weight=0.0)
grp_cp.calibrate(prob_cal, y_cal, s_cal)
sets_grp = grp_cp.predict_sets(prob_test, s_test)
m_grp = compute_all_metrics(sets_grp, y_test, s_test)

# 3. FairTransCP (Proposed Framework)
fair_cp = FairTransCP(alpha=alpha, score_fn='softmax', fairness_weight=0.4)
fair_cp.calibrate(prob_cal, y_cal, s_cal)
sets_fair = fair_cp.predict_sets(prob_test, s_test)
m_fair = compute_all_metrics(sets_fair, y_test, s_test)

# Comparison Table
df_res = pd.DataFrame([
    {
        'Method': 'Standard CP (Marginal)',
        'Worst-Group Coverage': f"{m_std['worst_group_coverage']*100:.1f}%",
        'Average Set Size': f"{m_std['avg_set_size']:.3f}",
        'Set-Size Disparity': f"{m_std['set_size_disparity']:.3f}",
        'Assessment': 'Leaves protected group under-covered'
    },
    {
        'Method': 'Group-Conditional CP',
        'Worst-Group Coverage': f"{m_grp['worst_group_coverage']*100:.1f}%",
        'Average Set Size': f"{m_grp['avg_set_size']:.3f}",
        'Set-Size Disparity': f"{m_grp['set_size_disparity']:.3f}",
        'Assessment': 'Spikes set-size disparity (The Paradox!)'
    },
    {
        'Method': 'FairTransCP (Our Method)',
        'Worst-Group Coverage': f"{m_fair['worst_group_coverage']*100:.1f}%",
        'Average Set Size': f"{m_fair['avg_set_size']:.3f}",
        'Set-Size Disparity': f"{m_fair['set_size_disparity']:.3f}",
        'Assessment': 'Optimal Pareto balance (High coverage + Low disparity)'
    }
])
df_res

### 📈 Step 4: Visualize the Disparity Spike and Mitigation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=130)

methods = ['Standard CP', 'Group CP', 'FairTransCP']
disparities = [m_std['set_size_disparity'], m_grp['set_size_disparity'], m_fair['set_size_disparity']]
coverages = [m_std['worst_group_coverage'], m_grp['worst_group_coverage'], m_fair['worst_group_coverage']]

# Plot 1: Set-Size Disparity
bars1 = ax1.bar(methods, disparities, color=['#94A3B8', '#EF4444', '#2563EB'], width=0.5, edgecolor='black')
ax1.axhline(1.0, color='gray', linestyle='--', label='Perfect Parity (1.0)')
ax1.set_ylabel('Set-Size Disparity Ratio (Lower is Better)', fontweight='bold')
ax1.set_title('Set-Size Disparity (1.0 = Ideal)', fontweight='bold')
for b in bars1:
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.003, f"{b.get_height():.3f}", ha='center', fontweight='bold')
ax1.legend()

# Plot 2: Worst-Group Coverage
bars2 = ax2.bar(methods, [c * 100 for c in coverages], color=['#94A3B8', '#EF4444', '#2563EB'], width=0.5, edgecolor='black')
ax2.axhline(90.0, color='green', linestyle=':', label='90% Target Guarantee')
ax2.set_ylabel('Worst-Group Coverage % (Higher is Better)', fontweight='bold')
ax2.set_title('Worst-Group Coverage (Target = 90%)', fontweight='bold')
for b in bars2:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3, f"{b.get_height():.1f}%", ha='center', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

### 👤 Step 5: Test on an Individual Loan Applicant
Inspect what prediction set is output for an applicant under each method:

In [ ]:
applicant_idx = 12
group = s_test[applicant_idx]
label_str = 'Good Credit (Repayment)' if y_test[applicant_idx] == 1 else 'Bad Credit (Risk/Default)'

labels = {0: 'Reject (Default Risk)', 1: 'Approve (Good Credit)'}

print('======================================================')
print(f"APPLICANT #{applicant_idx} EVALUATION")
print('======================================================')
print(f"Demographic Group:   {group}")
print(f"Ground Truth Status: {label_str}")
print(f"Model Probabilities: [Bad: {prob_test[applicant_idx][0]:.3f}, Good: {prob_test[applicant_idx][1]:.3f}]")
print('------------------------------------------------------')
print(f"Standard CP Prediction Set:     {[labels[i] for i in sets_std[applicant_idx]]}")
print(f"Group-Conditional CP Set:       {[labels[i] for i in sets_grp[applicant_idx]]}")
print(f"FairTransCP (Our Method) Set:   {[labels[i] for i in sets_fair[applicant_idx]]}")
print('======================================================')